# 09. Mapping & Outputs
**목적**: 최종 위험도 지도와 보고서용 Figure를 생성하고, 제출용 파일을 export한다.

산출물:
- Figure 1: 전력설비 분포 + 기상관측소
- Figure 2: 최종 위험도 지도 (정적, 보고서용)
- Figure 3: Top-5% 설비 우선점검 지도
- Interactive map: Folium 기반 HTML (코드 제출용)
- 최종 결과 테이블 CSV

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False
import folium
import warnings
warnings.filterwarnings('ignore')

from config import (DATA_PROCESSED, OUT_FIGURES, OUT_MAPS, OUT_TABLES,
                    CRS_GEO, TOP_K_PCTS)

with open(DATA_PROCESSED / 'column_map.json', encoding='utf-8') as f:
    cmap = json.load(f)
FC = cmap['facility']
FID_COL = FC['facility_id']

df_risk = pd.read_parquet(DATA_PROCESSED / 'risk_scores.parquet')
gdf_fac = gpd.read_file(DATA_PROCESSED / 'facility_geo.gpkg')   # WGS84
gdf_stn = gpd.read_file(DATA_PROCESSED / 'stations_proj.gpkg').to_crs(CRS_GEO)

# 설비별 평균 위험도
df_avg = df_risk.groupby(FID_COL).agg(
    final_risk_mean=('final_risk', 'mean'),
    final_risk_max=('final_risk', 'max'),
    risk_grade=('risk_grade', lambda x: x.value_counts().idxmax()),
    weather_contrib_mean=('weather_hazard', 'mean'),
    spatial_contrib_mean=('spatial_exposure', 'mean'),
    facility_contrib_mean=('facility_exposure', 'mean'),
).reset_index()

gdf_plot = gdf_fac.merge(df_avg, on=FID_COL, how='left')
print(f'지도용 GeoDataFrame: {len(gdf_plot):,}개 설비')

GRADE_COLORS = {
    'Very High': '#d62728',
    'High':      '#ff7f0e',
    'Moderate':  '#f7c948',
    'Low':       '#2ca02c',
}

OUT_FIGURES.mkdir(parents=True, exist_ok=True)
OUT_MAPS.mkdir(parents=True, exist_ok=True)

## Figure 1: 전력설비 분포 + 관측소

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))

gdf_plot.plot(ax=ax, color='steelblue', markersize=3, alpha=0.6, label='전력설비')
gdf_stn.plot(ax=ax, color='red', marker='^', markersize=30, alpha=0.8, label='기상관측소')

ax.set_title('전력설비 및 기상관측소 분포', fontsize=14)
ax.set_xlabel('경도', fontsize=11)
ax.set_ylabel('위도', fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_FIGURES / 'fig1_facility_station_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 저장')

## Figure 2: 최종 위험도 지도 (보고서용)

In [ ]:
gdf_plot['color'] = gdf_plot['risk_grade'].map(GRADE_COLORS).fillna('#aec6e8')
gdf_plot['markersize'] = gdf_plot['final_risk_mean'].apply(
    lambda x: 3 if x < 50 else (6 if x < 66 else (10 if x < 86 else 15))
)

fig, ax = plt.subplots(figsize=(11, 13))

for grade, color in GRADE_COLORS.items():
    subset = gdf_plot[gdf_plot['risk_grade'] == grade]
    if len(subset) > 0:
        subset.plot(ax=ax, color=color,
                    markersize=subset['markersize'],
                    alpha=0.8, label=f'{grade} ({len(subset):,}개)')

legend_patches = [
    mpatches.Patch(color=v, label=k) for k, v in GRADE_COLORS.items()
]
ax.legend(handles=legend_patches, title='위험등급', fontsize=11, title_fontsize=11)
ax.set_title('전력설비 화재위험도 지도 (WPFI)', fontsize=15)
ax.set_xlabel('경도', fontsize=11)
ax.set_ylabel('위도', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_FIGURES / 'fig2_final_risk_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 저장')

## Figure 3: Top-5% 우선점검 지도

In [ ]:
k5 = max(1, int(len(gdf_plot) * 0.05))
gdf_top5 = gdf_plot.nlargest(k5, 'final_risk_mean')

fig, ax = plt.subplots(figsize=(11, 13))

# 전체 설비 (회색)
gdf_plot.plot(ax=ax, color='#cccccc', markersize=2, alpha=0.4, label='일반 설비')

# Top-5% 고위험 설비
gdf_top5.plot(ax=ax, color='#d62728', markersize=12, alpha=0.9,
              label=f'Top 5% 고위험 설비 ({k5:,}개)')

# 상위 10개 설비에 번호 annotation
for i, (_, row) in enumerate(gdf_top5.head(10).iterrows()):
    ax.annotate(
        f'{i+1}',
        xy=(row.geometry.x, row.geometry.y),
        xytext=(5, 5), textcoords='offset points',
        fontsize=9, color='black',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7)
    )

ax.set_title(f'우선점검 대상 설비 (Top 5%, {k5:,}개)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_FIGURES / 'fig3_topk_inspection_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 3 저장')

## Interactive Map (Folium)

In [ ]:
# 중심 좌표 계산
center_lat = gdf_plot.geometry.y.mean()
center_lon = gdf_plot.geometry.x.mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=7,
               tiles='CartoDB positron')

# 위험등급별 레이어 그룹
grade_order = ['Very High', 'High', 'Moderate', 'Low']
for grade in grade_order:
    color = GRADE_COLORS[grade]
    subset = gdf_plot[gdf_plot['risk_grade'] == grade]
    if len(subset) == 0:
        continue

    layer = folium.FeatureGroup(name=f'{grade} ({len(subset):,}개)')
    radius = {'Very High': 8, 'High': 6, 'Moderate': 4, 'Low': 3}[grade]

    for _, row in subset.iterrows():
        popup_text = (
            f"<b>{FID_COL}:</b> {row[FID_COL]}<br>"
            f"<b>위험도:</b> {row.get('final_risk_mean', 0):.1f}<br>"
            f"<b>등급:</b> {grade}<br>"
            f"<b>기상위험:</b> {row.get('weather_contrib_mean', 0):.1f}<br>"
            f"<b>공간노출:</b> {row.get('spatial_contrib_mean', 0):.1f}<br>"
            f"<b>설비노출:</b> {row.get('facility_contrib_mean', 0):.1f}"
        )
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            popup=folium.Popup(popup_text, max_width=250)
        ).add_to(layer)

    layer.add_to(m)

folium.LayerControl().add_to(m)

map_path = OUT_MAPS / 'wpfi_interactive_map.html'
m.save(str(map_path))
print(f'Interactive map 저장: {map_path}')

## 최종 결과 테이블 Export

In [ ]:
# 설비별 최종 결과 (보고서 테이블용)
df_final = gdf_plot.drop(columns=['geometry', 'color', 'markersize']).copy()
df_final = df_final.sort_values('final_risk_mean', ascending=False).reset_index(drop=True)
df_final['inspection_rank'] = df_final.index + 1

# Top-K coverage 계산
total_risk = df_final['final_risk_mean'].sum()
print('=== Top-K Risk Coverage ===')
for pct in TOP_K_PCTS:
    k = max(1, int(len(df_final) * pct))
    coverage = df_final.head(k)['final_risk_mean'].sum() / total_risk * 100
    print(f'Top {int(pct*100)}% ({k:,}개) → 전체 누적 위험도의 {coverage:.1f}% 커버')

df_final.to_csv(OUT_TABLES / 'final_risk_table.csv', index=False)
print('\nfinal_risk_table.csv 저장')

# 보고서 표 (상위 20개)
report_cols = [FID_COL, 'inspection_rank', 'final_risk_mean', 'risk_grade',
               'weather_contrib_mean', 'spatial_contrib_mean', 'facility_contrib_mean']
report_cols = [c for c in report_cols if c in df_final.columns]
df_final[report_cols].head(20).to_csv(OUT_TABLES / 'report_top20.csv', index=False)
print('report_top20.csv 저장')
print('\n=== 전체 파이프라인 완료 ===')